# Notebook 5: Evaluation & Decoding Strategies
Đánh giá độc lập 3 mô hình đã train (BARTpho Full FT, BARTpho LoRA, Qwen2.5 LoRA).
YÊU CẦU: BẬT GPU T4 x2 TRÊN KAGGLE

In [ ]:
!pip install transformers datasets evaluate rouge_score bert_score peft tqdm "torchao>=0.16.0"

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
from peft import PeftModel
from evaluate import load
from tqdm import tqdm
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Tải test set
test_df = pd.read_csv("test_1k.csv")
articles = test_df["article"].tolist()
references = test_df["abstract"].tolist()

# Tải metrics
rouge = load("rouge")

In [ ]:
# 1. Hàm Inference
def generate_summaries_seq2seq(model, tokenizer, texts, decoding_strategy="beam"):
    summaries = []
    eval_texts = texts[:100] # Lấy 100 mẫu demo
    
    for text in tqdm(eval_texts, desc=f"Generating Seq2Seq ({decoding_strategy})"):
        inputs = tokenizer(text, max_length=512, truncation=True, return_tensors="pt").to(device)
        
        if decoding_strategy == "beam":
            outputs = model.generate(**inputs, max_length=128, num_beams=4, early_stopping=True)
        elif decoding_strategy == "sampling_top_p":
            outputs = model.generate(**inputs, max_length=128, do_sample=True, top_p=0.9, top_k=50)
            
        summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
        summaries.append(summary)
    return summaries

def generate_summaries_causal(model, tokenizer, texts, decoding_strategy="beam"):
    summaries = []
    eval_texts = texts[:100] # Lấy 100 mẫu demo
    
    for text in tqdm(eval_texts, desc=f"Generating CausalLM ({decoding_strategy})"):
        prompt = f"Tóm tắt bài báo sau:\n{text}\n\nTóm tắt:\n"
        inputs = tokenizer(prompt, max_length=1024, truncation=True, return_tensors="pt").to(device)
        
        # Lấy độ dài của prompt để cắt đi phần prompt trong output
        prompt_length = inputs["input_ids"].shape[1]
        
        if decoding_strategy == "beam":
            outputs = model.generate(**inputs, max_new_tokens=128, num_beams=4, early_stopping=True, pad_token_id=tokenizer.eos_token_id)
        elif decoding_strategy == "sampling_top_p":
            outputs = model.generate(**inputs, max_new_tokens=128, do_sample=True, top_p=0.9, top_k=50, pad_token_id=tokenizer.eos_token_id)
            
        # Chỉ lấy phần sinh ra mới
        summary_ids = outputs[0][prompt_length:]
        summary = tokenizer.decode(summary_ids, skip_special_tokens=True)
        summaries.append(summary)
    return summaries

def evaluate_summaries(predictions, refs):
    actual_refs = refs[:len(predictions)]
    r_score = rouge.compute(predictions=predictions, references=actual_refs)
    return {"ROUGE-1": r_score['rouge1'], "ROUGE-L": r_score['rougeL']}

# Biến lưu kết quả chung
all_results = pd.DataFrame({"article": articles, "reference": references})

In [ ]:
# 2. Đánh giá BARTpho Full FT 
print("\n--- ĐÁNH GIÁ MÔ HÌNH BARTpho FULL FT ---")
FULL_FT_PATH = "./bartpho_full_ft_final" 
if os.path.exists(FULL_FT_PATH):
    tokenizer_full = AutoTokenizer.from_pretrained(FULL_FT_PATH)
    model_full = AutoModelForSeq2SeqLM.from_pretrained(FULL_FT_PATH).to(device)

    preds_full_beam = generate_summaries_seq2seq(model_full, tokenizer_full, articles, "beam")
    print("KQ Full FT (Beam):", evaluate_summaries(preds_full_beam, references))
    all_results["bartpho_full_beam"] = preds_full_beam

    del model_full
    torch.cuda.empty_cache()
else:
    print("Không tìm thấy model BARTpho Full FT. Bỏ qua.")

In [ ]:
# 3. Đánh giá BARTpho LoRA
print("\n--- ĐÁNH GIÁ MÔ HÌNH BARTpho LORA ---")
LORA_PATH = "./bartpho_lora_final"
BASE_MODEL_NAME = "vinai/bartpho-syllable"
if os.path.exists(LORA_PATH):
    base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_NAME)
    model_lora = PeftModel.from_pretrained(base_model, LORA_PATH).to(device)
    tokenizer_lora = AutoTokenizer.from_pretrained(LORA_PATH)

    preds_lora_beam = generate_summaries_seq2seq(model_lora, tokenizer_lora, articles, "beam")
    print("KQ BARTpho LoRA (Beam):", evaluate_summaries(preds_lora_beam, references))
    all_results["bartpho_lora_beam"] = preds_lora_beam
    
    del model_lora
    torch.cuda.empty_cache()
else:
    print("Không tìm thấy model BARTpho LoRA. Bỏ qua.")

In [ ]:
# 4. Đánh giá Qwen2.5 LoRA
print("\n--- ĐÁNH GIÁ MÔ HÌNH Qwen2.5 LORA ---")
QWEN_LORA_PATH = "./qwen-lora-vietnews"
QWEN_BASE = "Qwen/Qwen2.5-0.5B"
if os.path.exists(QWEN_LORA_PATH):
    base_qwen = AutoModelForCausalLM.from_pretrained(QWEN_BASE, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
    model_qwen = PeftModel.from_pretrained(base_qwen, QWEN_LORA_PATH)
    tokenizer_qwen = AutoTokenizer.from_pretrained(QWEN_BASE, trust_remote_code=True)
    if tokenizer_qwen.pad_token is None:
        tokenizer_qwen.pad_token = tokenizer_qwen.eos_token

    preds_qwen_beam = generate_summaries_causal(model_qwen, tokenizer_qwen, articles, "beam")
    print("KQ Qwen LoRA (Beam):", evaluate_summaries(preds_qwen_beam, references))
    all_results["qwen_lora_beam"] = preds_qwen_beam
    
    del model_qwen
    torch.cuda.empty_cache()
else:
    print("Không tìm thấy model Qwen LoRA. Bỏ qua.")

In [ ]:
# 5. Phục vụ Error Analysis (Xuất ra CSV)
all_results.to_csv("all_models_predictions.csv", index=False)
print("Đã lưu kết quả sinh của tất cả các mô hình vào all_models_predictions.csv!")